# App 7 · LLMOps — 让钱、延迟、错误都看得见

凌晨 3 点收到告警：过去一小时 token 消耗暴涨 40 倍。打开后台一看，QPS 没异常、用户没投诉。问题在哪？

如果你的 LLM 系统没有 trace，这个问题没法答。token 消耗是个标量，告诉你"花了多少钱"但不告诉你"花在哪一步"。可能是某个 prompt 模板被人改坏了，把 `{context}` 写成了 `{{context}}`，导致每次都把整个文档库塞进 prompt；可能是 retry 逻辑死循环；可能是某个 RAG 配置 chunk_size 写错了。这些根因，没有 span tree 就只能靠猜。

LLM 系统的可观测性比普通 web 后端难，难在四点：**每次调用花钱**（不只是延迟）、**延迟天然慢**（看 p95/p99 而不是 avg）、**失败模式多**（hallucination、限流、超时、bad output）、**质量难量化**（HTTP 200 不代表答对）。这一节我们用 `utils/observability.py` 的 `@observe` 装饰器和 `with span()` 上下文管理器把这四个维度接到 trace 树里——同样的 API 在没装 Langfuse 时走 in-memory 的 `MockObserver`，装了 Langfuse 时自动切到真后端。

> **跑这一节前**：跑过 App5 / App6（理解 MCP 工具调用和 Skill 路由——这些都是值得追踪的"能产生 span 的事件"）。可选 SDK `pip install 'langfuse>=2.0'` 装上能看到真 dashboard，不装就走 mock。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"📂 repo root: {_root}")


📂 repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


## 1. LLMOps 三支柱：Trace、Metric、Alert

把 LLM 系统接到现代 SRE 工具栈，要回答的问题分成三类：

| 支柱 | 能回答什么 | 数据形态 | 工具栈 |
|---|---|---|---|
| **Trace** | 这条请求经过了哪些 LLM/tool？哪步慢？ | 树形 span（一次请求一棵树） | Langfuse / LangSmith / OpenTelemetry |
| **Metric** | 整体 QPS / token / p95 延迟？ | 标量时间序列 | Prometheus / Datadog |
| **Alert** | 失败率突变？token 暴涨？ | 阈值规则 + 通知 | PagerDuty / 企业微信 |

三者**用同一份原始数据**——每次 LLM 调用产生一棵 span 树（trace）；把树的某些字段聚合就是 metric；metric 触阈值就触发 alert。所以**先把 trace 做对，metric 和 alert 是自然而然的派生品**。这就是为什么这一节先讲 trace。

形式上，一个 trace 是一棵有根树 $T = (V, E, r)$：

- $V$ 是 span 的集合，每个 span $v_i$ 携带 `(name, start_time, end_time, input, output, metadata)`
- $E \subseteq V \times V$ 是父子关系——$v_j$ 是 $v_i$ 的子 span 当且仅当 $v_j$ 在 $v_i$ 的执行期间被创建
- $r$ 是 root span，对应一次完整的用户请求

把 LLM 调用、tool 调用、RAG 检索、Multi-Agent 消息全都包成 span 加进这棵树，你就拿到了"这条请求做了什么"的完整重放。

In [2]:
from utils.observability import observer, observe, span, get_backend
import time, json

print(f"backend: {get_backend()} (mock=本地内存; langfuse=已配 LANGFUSE_*)")


backend: mock (mock=本地内存; langfuse=已配 LANGFUSE_*)


## 2. `@observe` 装饰器：一行接入

装饰器是把"加 trace"和"业务逻辑"解耦的最干净方式。`@observe` 在函数前后做这些事——记录开始时间、抓 input、调用原函数、抓 output、记录结束时间、估 token 数、把 span 挂到当前 trace 树上。

整个过程对业务函数透明——它甚至不知道自己被 trace 了。这种**侵入性极低**的设计是装饰器模式的核心价值，也是为什么 OpenTelemetry、Sentry、Datadog 在 Python 生态都首选这个 API 风格。

下面看 `utils/observability.py` 的实现。注意 backend 检测：装了 Langfuse 且配了 env var 就走真上报，否则走 in-memory 的 `MockObserver`——同一份业务代码，开发期看 mock trace、生产期上报 dashboard，零改动。

In [3]:
observer.reset()

@observe("retrieve")
def retrieve(query):
    time.sleep(0.05)  # 模拟检索延迟
    return f"docs for '{query}'"

@observe("generate")
def generate(query, docs):
    time.sleep(0.1)
    return f"Answer based on {docs[:30]}"

@observe("rag_pipeline")
def rag(query):
    docs = retrieve(query)
    return generate(query, docs)

# 跑 3 次
for q in ["什么是 LoRA?", "RAG 怎么工作?", "Skills 是什么?"]:
    rag(q)

print("Trace tree:")
observer.print_tree()
print("\nSummary:")
print(json.dumps(observer.summary(), indent=2, ensure_ascii=False))


Trace tree:
├─ rag_pipeline (165ms)
  ├─ retrieve (56ms)
  ├─ generate (109ms)
├─ rag_pipeline (171ms)
  ├─ retrieve (62ms)
  ├─ generate (109ms)
├─ rag_pipeline (171ms)
  ├─ retrieve (62ms)
  ├─ generate (109ms)

Summary:
{
  "n_traces": 3,
  "n_total_spans": 9,
  "total_duration_ms": 506.7,
  "tokens": {
    "prompt": 47,
    "completion": 91,
    "total": 138
  },
  "cost_usd": 0.0001
}


> **关于 token 估算**：本 demo 用 `len(repr(obj)) // 3` 做近似估算——中文每字 ~1.3 token、英文 ~0.25 token、JSON/code 介于之间，整体平均接近 1/3。这够看大致量级、不够做计费。生产精确计数请用 `tiktoken`：
>
> ```python
> import tiktoken
> enc = tiktoken.encoding_for_model("gpt-4")
> tokens = len(enc.encode(text))
> ```
>
> 真接 Langfuse 时它会从 LLM 响应的 `usage.prompt_tokens / completion_tokens` 字段里读，比估算更准——毕竟 OpenAI/Anthropic 自己知道。

## 3. `with span()`：手动控制 + 跨 Agent 透传

`@observe` 是函数级的——一个函数一个 span。但有时候你想在函数内部分多个细粒度 span（比如"检索"和"生成"是同一个函数里的两步，但要分开看耗时），这时候就要用 `with span(...)` 上下文管理器。

更重要的用途是**跨 Agent 透传**。Multi-Agent 系统里，一次用户请求会经过 Planner → Worker → Reviewer 三个 agent，每个 agent 自己有内部 span。如果不做透传，你拿到的是三棵独立的 trace 树，看不出"这个 worker 调用是属于哪个 user request"。

解法是在 user request 入口起一个 root span，让所有下游 agent 的 span 自然嵌套进去——`with span("user_request") as root:` 一行搞定。这跟 OpenTelemetry 的 [trace context propagation](https://www.w3.org/TR/trace-context/) 是同一回事，只是简化版。

In [4]:
observer.reset()

@observe("planner.run")
def planner_run(task):
    return f"Plan: split {task} into 2 subtasks"

@observe("worker.run")
def worker_run(plan):
    time.sleep(0.05)
    return f"Worker did: {plan[:30]}"

@observe("reviewer.review")
def reviewer_review(result):
    return f"APPROVE: {result[:20]}"

# 用 with span 包整个 user request — 形成嵌套 trace tree
with span("user_request", input={"task": "写一个缓存装饰器"}) as root:
    plan = planner_run("写一个缓存装饰器")
    result = worker_run(plan)
    review = reviewer_review(result)

print("Multi-Agent Trace tree (含 root span):")
observer.print_tree()

# 找出最慢的 span
def collect(s, out, depth=0):
    if s.duration_ms is not None:
        out.append((s.name, s.duration_ms, depth))
    for c in s.children:
        collect(c, out, depth + 1)
all_spans = []
for r in observer.spans:
    collect(r, all_spans)
all_spans.sort(key=lambda x: -x[1])
print(f"\n最慢的 3 个 span:")
for name, ms, depth in all_spans[:3]:
    print(f"  {ms:>7.1f}ms  {'  '*depth}{name}")


Multi-Agent Trace tree (含 root span):
├─ user_request (57ms)
  ├─ planner.run (running)
  ├─ worker.run (57ms)
  ├─ reviewer.review (running)

最慢的 3 个 span:
     57.1ms  user_request
     57.1ms    worker.run
      0.0ms    planner.run


## 4. Token 估算 + 成本可见

LLM 系统比 web 后端"贵"得多——一次完整 RAG 调用动辄几千 token，按 GPT-4 价格算就是几美分。如果 token 用量看不见，**你不知道一个 query 烧了多少钱、哪个 prompt 模板成本高、哪条链路有优化空间**。

`@observe` 自动累计每次调用的 input/output token 估算 + 按 mock pricing 算成本。在 `MockObserver.summary()` 里你能看到一个简单的汇总——n_traces / n_total_spans / 总耗时 / token / cost_usd。生产系统会把这套数据接到 dashboard，做 per-user / per-tenant / per-feature 的成本归因。

下面跑一次看。

In [5]:
print("过去 N 个请求累计 trace summary:")
print(json.dumps(observer.summary(), indent=2, ensure_ascii=False))

print(f"""
💡 生产场景:
  - tokens 看不见 = 不知道烧了多少钱
  - p95 延迟看不见 = 不知道用户体验
  - 错路由看不见 = 优化无从下手

LLMOps 是把 ML/Agent 系统接到现代 SRE 工具栈的桥梁。
""")


过去 N 个请求累计 trace summary:
{
  "n_traces": 1,
  "n_total_spans": 4,
  "total_duration_ms": 57.1,
  "tokens": {
    "prompt": 28,
    "completion": 35,
    "total": 63
  },
  "cost_usd": 0.0001
}

💡 生产场景:
  - tokens 看不见 = 不知道烧了多少钱
  - p95 延迟看不见 = 不知道用户体验
  - 错路由看不见 = 优化无从下手

LLMOps 是把 ML/Agent 系统接到现代 SRE 工具栈的桥梁。



In [ ]:
# 自检：trace tree / token 估算 / 嵌套 span 是否都体验过了
def verify_app7() -> bool:
    print("=" * 56)
    print("自检 · App7 LLMOps 可观测性")
    print("=" * 56)
    checks: list[tuple[str, bool, str]] = []

    try:
        summary = observer.summary()  # noqa: F821 —— utils.observability 全局单例
        checks.append(("observer 收到 ≥1 个 trace", summary["n_traces"] >= 1,
                       f"n_traces={summary['n_traces']}"))
        checks.append(("trace 包含 ≥3 个 span（嵌套结构）",
                       summary["n_total_spans"] >= 3,
                       f"n_total_spans={summary['n_total_spans']}"))
        checks.append(("token 估算非零（@observe 自动累计）",
                       summary["tokens"]["total"] > 0,
                       f"total={summary['tokens']['total']}"))
        has_nested = any(len(root.children) > 0 for root in observer.spans)  # noqa: F821
        checks.append(("trace tree 有真嵌套（root → child）",
                       has_nested,
                       "嵌套生效" if has_nested else "全是平级"))
    except (NameError, AttributeError, KeyError):
        checks.append(("observer 可用", False, "⏭ §2 / §3 cell 未跑或类型异常"))

    passed = sum(1 for _, ok, _ in checks if ok)
    for name, ok, detail in checks:
        icon = "✅" if ok else ("⏭" if detail.startswith("⏭") else "❌")
        print(f"  {icon} {name}  ({detail})")
    print(f"\n通过 {passed}/{len(checks)}")
    if passed == len(checks):
        print("下一节：App8_Production_Capstone")
    elif passed >= 2:
        print("部分通过——把跳过的 cell 跑完后重跑这一格。")
    else:
        print("未通过——回到顶部按顺序 Run All。")
    return passed == len(checks)


verify_app7()


## 5. 收尾：从 Mock 到 Langfuse

如果你想把这套接到真 Langfuse dashboard：

```bash
# 1. 注册 https://langfuse.com 或本地起 docker
docker compose up langfuse

# 2. 设置 env var
export LANGFUSE_PUBLIC_KEY=pk_...
export LANGFUSE_SECRET_KEY=sk_...
export LANGFUSE_HOST=http://localhost:3000

# 3. 重跑——utils/observability.py 自动检测，@observe 走真后端
```

之后 mock 的 trace 树会上报到 Langfuse dashboard，能看到：每条对话的完整 trace + token + 延迟、Top-N 慢路径、失败率突变告警。

---

走完这一节你应该明白三件事：

**第一，trace 是源头**——metric 和 alert 都是 trace 数据的聚合派生。先把每次 LLM 调用都包进 span 树，后面想加 metric 加 alert 都很轻。

**第二，装饰器 + 上下文管理器是接入 trace 的最干净 API**。`@observe` 和 `with span()` 的组合覆盖了 90% 的 trace 需求，业务代码几乎不用改。

**第三，token 看不见 = 钱看不见**。LLM 系统的成本分析是 web 后端没有的新维度，必须从一开始就埋进 trace。

下一节 [App8 Capstone](./App8_Production_Capstone.ipynb) 把前面 7 节的全部能力（Multi-Agent / MCP / RAG / Skills / LLMOps）合成一个 production pipeline，跑 batch eval 看综合表现。